## Content Understanding - "Layout" API for Invoice Analysis

![content-understanding](../Images/content-understanding.png)

### Installing Utilities and Libraries

In [ ]:
%pip install python-dotenv azure-ai-contentunderstanding==1.1.0 azure-identity

### Setting up the Environment Variables

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput, AnalysisResult
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import AzureError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import os
import json
import sys

load_dotenv()

endpoint = os.getenv("FOUNDRY_AI_SERVICES_ENDPOINT")
api_key = os.getenv("FOUNDRY_AI_SERVICES_KEY")
api_version = "2025-11-01"
analyzer_id = "prebuilt-layout"

print("Endpoint:", endpoint)
print("API Key:", api_key)

### Initializing the Content Understanding Client

In [ ]:
# Set up Content Understanding client.
credential = AzureKeyCredential(api_key) if api_key and "{{FOUNDRY_AI_SERVICES_KEY}}" not in api_key else DefaultAzureCredential()
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

### Analyze Invoice using the "Layout" API

In [ ]:
# Set the file URL
file_url = "https://raw.githubusercontent.com/kuljotSB/AI-901-Azure-AI-Fundamentals/main/Content-Understanding/Prebuilt-Layout-API/Samples/invoice.pdf"

# Start analyzing an invoice using the "LAYOUT" API
print("\nAnalyzing invoice using the 'LAYOUT' API...")

try:
    poller = client.begin_analyze(
        analyzer_id=analyzer_id,
        inputs = [AnalysisInput(url=file_url)]
    )

    result: AnalysisResult = poller.result()

except AzureError as err:
    print("Error analyzing document:", {err.message})
    sys.exit(1)
except Exception as ex:
    print("An unexpected error occurred:", {str(ex)})
    sys.exit(1)

### Display Results for Invoice Analysis

In [ ]:
# [START output_result]
print("=" * 50)
print("Analysis result:")
print("=" * 50 + "\n")

result_str = json.dumps(result.as_dict(), indent=2)
ret_lines = result_str.splitlines()

print("\n".join(ret_lines[:100]))  # Print only the first 100 lines for brevity

# Extract the markdown content and create a .md file
markdown_content = json.loads(result_str)["contents"][0]["markdown"]
with open("invoice_analysis_result.md", "w") as md_file:
    md_file.write(markdown_content)
# [END output_result]